# Explainability Pipeline: GNN + SHAP + Autoencoder + LLM

This notebook generates the explanation artifacts consumed by the Streamlit web app:

1. **SHAP values** for the LightGBM ranker (interpretable features only — no latent PCA axes)
2. **GNNExplainer** node-level importance scores (aggregated from edge masks onto hub nodes)
3. **Autoencoder anomaly transactions** with risk scores for colour-coding in the UI
4. **Batch LLM explanations** for medium/high-risk customers (deterministic fallback for low-risk)
5. **Unified bundle** consumed by the Streamlit Model Output and Run Model pages

## 1. Setup & Load Artifacts

In [ ]:
from __future__ import annotations

import json
import os
import time
from pathlib import Path

import joblib
import lightgbm as lgb
import numpy as np
import pandas as pd
import shap
import torch

from lib.explainability import build_transaction_ae_features, load_temporal_autoencoder
from lib.model_utils import load_artifacts, load_sage_model
from lib.resource_paths import OUTPUTS_DIR, DATA_DIR
from lib.llm_config import load_llm_config, test_endpoint, call_llm, assemble_prompt

try:
    from torch_geometric.data import HeteroData
    from torch_geometric.explain import Explainer, GNNExplainer
    from torch_geometric.loader import NeighborLoader
    import torch_geometric.transforms as T
    TORCH_GEO_OK = True
except Exception:
    TORCH_GEO_OK = False

OUTPUTS_DIR = OUTPUTS_DIR
paths = {
    'rank_df': OUTPUTS_DIR / 'rank_df_with_anchor_expansion.csv.gz',
    'model_output': OUTPUTS_DIR / 'model_output.csv',
    'sage_model': OUTPUTS_DIR / 'fraud_sage_model.pth',
    'sage_artifacts': OUTPUTS_DIR / 'sage_artifacts.pkl',
    'lgbm': OUTPUTS_DIR / 'lgbm_fraud_ranker.joblib',
    'ae_scores': OUTPUTS_DIR / 'autoencoder_transaction_scores.csv.gz',
    'master_pool': OUTPUTS_DIR / 'master_transaction_pool.csv.gz',
}

missing = [k for k, p in paths.items() if not p.exists()]
assert not missing, f'Missing required artifacts: {missing}'

rank_df = pd.read_csv(paths['rank_df'])
model_output = pd.read_csv(paths['model_output'])
ae_scores = pd.read_csv(paths['ae_scores'])
master_pool = pd.read_csv(paths['master_pool'], compression='gzip', low_memory=False)

for col in ['customer_id']:
    for df in [rank_df, model_output, ae_scores, master_pool]:
        if col in df.columns:
            df[col] = df[col].astype(str)

lgb_model = joblib.load(paths['lgbm'])

print('Artifacts loaded from:', OUTPUTS_DIR)
print(f'  rank_df: {len(rank_df):,} rows')
print(f'  model_output: {len(model_output):,} rows')
print(f'  master_pool: {len(master_pool):,} rows')
print(f'  TORCH_GEO_OK: {TORCH_GEO_OK}')

## 2. LLM Endpoint Configuration

Loads settings from `configs/llm_config.yaml`. Tests connectivity if enabled.

When **LLM is disabled**, all customers get deterministic explanations (rule-based from SHAP values).
When **LLM is enabled**, medium/high-risk customers get LLM explanations; low-risk get deterministic.

In [ ]:
llm_config = load_llm_config()
print(f'LLM enabled: {llm_config.enabled}')
print(f'Endpoint:    {llm_config.endpoint}')
print(f'Format:      {llm_config.request_format}')
print(f'Model:       {llm_config.model}')

if llm_config.enabled:
    llm_test = test_endpoint(llm_config)
    print(f'\nEndpoint test: {"SUCCESS" if llm_test["success"] else "FAILED"}')
    print(f'  Model:  {llm_test.get("model")}')
    print(f'  Latency: {llm_test.get("latency_sec", "?")}s')
    if llm_test.get("error"):
        print(f'  Error:  {llm_test["error"]}')
    if llm_test.get("sample"):
        print(f'  Sample: {llm_test["sample"]}')
else:
    print('\nLLM disabled — all explanations will use deterministic fallback.')

# Save test result
OUTPUTS_DIR.joinpath('llm_endpoint_test.json').write_text(
    json.dumps(llm_test if llm_config.enabled else {
        'success': False, 'error': 'LLM not enabled'
    }, indent=2)
)

## 3. SHAP Values for LightGBM

Computes SHAP feature attributions for all customers.

**Important:** The embedding PCA features (`emb_pca_*`) are excluded — they're uninterpretable latent axes from the GNN embedding. Only behavioural and ensemble features are kept so the LLM/deterministic explanation reads naturally.

In [ ]:
# ── All LGB features (model expects 33, including latent PCA axes) ──────────ALL_LGB_FEATURES = [    'emb_pca_1','emb_pca_2','emb_pca_3','emb_pca_4',    'emb_pca_5','emb_pca_6','emb_pca_7','emb_pca_8',    'km_component_size','km_component_train_fraud_rate','km_component_mean_dgi',    'hdb_component_size','hdb_component_fraud_rate_labeled','hdb_component_fraud_lift_labeled',    'knn_mean_distance','knn_suspicious_share','knn_gold_fraud_count',    'dist_to_fraud_centroid','dist_to_legit_centroid','centroid_margin',    'dgi_anomaly_score','customer_ae_risk_norm','gmm_max_prob','component_confidence',    'mlp_fraud_prob','cluster_consensus_score',    'hdb_outlier_score','min_dist_to_fraud_anchor','mean_dist_to_fraud_anchor',    'min_dist_to_legit_anchor','anchor_proximity_score',    'eft_amount_match_count','abm_dc_colocated',]ALL_LGB_FEATURES = [c for c in ALL_LGB_FEATURES if c in rank_df.columns]# Interpretable subset (no latent PCA axes — used for explanations)INTERPRETABLE_FEATURES = sorted(set(ALL_LGB_FEATURES) - {f'emb_pca_{i}' for i in range(1, 9)})print(f'Using {len(ALL_LGB_FEATURES)} total LGB features ({len(INTERPRETABLE_FEATURES)} interpretable)')print(f'Excluded: emb_pca_1..8 (latent GNN embedding axes)')rdf = rank_df.set_index('customer_id')X_all = rdf[ALL_LGB_FEATURES].apply(pd.to_numeric, errors='coerce').fillna(0.0)print(f'SHAP matrix shape: {X_all.shape}')# Compute SHAP values (or load from cache)shap_cache_path = OUTPUTS_DIR / 'lgbm_shap_webapp.jsonl'if shap_cache_path.exists():    print(f'Loading cached SHAP values from {shap_cache_path}')    all_shap_records = [json.loads(line) for line in open(shap_cache_path) if line.strip()]    print(f'Loaded {len(all_shap_records)} cached SHAP records')else:    print('Computing SHAP values (this may take a minute)...')    t0 = time.time()    expl = shap.TreeExplainer(lgb_model)    shap_vals = expl.shap_values(X_all.values)    if isinstance(shap_vals, list):        shap_vals = shap_vals[1]    print(f'SHAP computed in {time.time() - t0:.1f}s')    # Build SHAP records for every customerif not shap_cache_path.exists() or 'all_shap_records' not in dir() or len(all_shap_records) == 0:    # Build SHAP records from scratch    FEATURE_LABELS = {        'km_component_size': 'Cluster size',        'km_component_train_fraud_rate': 'Cluster fraud rate',        'km_component_mean_dgi': 'Cluster mean graph anomaly',        'hdb_component_size': 'HDB cluster size',        'hdb_component_fraud_rate_labeled': 'HDB fraud rate',        'hdb_component_fraud_lift_labeled': 'HDB fraud lift',        'knn_mean_distance': 'KNN mean distance',        'knn_suspicious_share': 'KNN suspicious share',        'knn_gold_fraud_count': 'KNN gold fraud count',        'dist_to_fraud_centroid': 'Distance to fraud centroid',        'dist_to_legit_centroid': 'Distance to legit centroid',        'centroid_margin': 'Centroid margin',        'dgi_anomaly_score': 'Graph anomaly score',        'customer_ae_risk_norm': 'Transaction anomaly risk',        'gmm_max_prob': 'Cluster membership confidence',        'component_confidence': 'Component confidence',        'mlp_fraud_prob': 'MLP fraud probability',        'cluster_consensus_score': 'Cluster consensus score',        'hdb_outlier_score': 'HDB outlier score',        'min_dist_to_fraud_anchor': 'Min dist to fraud anchor',        'mean_dist_to_fraud_anchor': 'Mean dist to fraud anchor',        'min_dist_to_legit_anchor': 'Min dist to legit anchor',        'anchor_proximity_score': 'Anchor proximity score',        'eft_amount_match_count': 'EFT amount matches',        'abm_dc_colocated': 'ABM/DC co-location',    }    all_shap_records = []    for i, (cid, row) in enumerate(X_all.iterrows()):        sv = shap_vals[i]        top_idx = np.argsort(np.abs(sv))[::-1][:10]        top_feats = [        {            'feature': ALL_LGB_FEATURES[j],            'description': FEATURE_LABELS.get(ALL_LGB_FEATURES[j], ALL_LGB_FEATURES[j]),            'shap_value': float(sv[j]),            'feature_value': float(X_all.values[i, j]),        }        for j in top_idx        ]        all_shap_records.append({        'customer_id': cid,        'lgb_fraud_prob': float(lgb_model.predict_proba(X_all.values[i:i+1])[0, 1]),        'base_value': float(            expl.expected_value if not isinstance(expl.expected_value, (list, np.ndarray))            else expl.expected_value[1]        ),        'top_shap_features': top_feats,        })# Save full SHAP recordsshap_jsonl_path = OUTPUTS_DIR / 'lgbm_shap_webapp.jsonl'with open(shap_jsonl_path, 'w') as f:    for r in all_shap_records:        f.write(json.dumps(r) + '\n')print(f'Saved: {shap_jsonl_path}')print(f'Records: {len(all_shap_records)}')

## 4. GNNExplainer: Node-Level Importance

Runs GNNExplainer on customer subgraphs and **aggregates edge importance to hub nodes** (merchant categories and cities).

Since each edge represents a transaction, and there are many transactions between a customer and a given category/city, edge-level importance doesn't make sense on its own. Instead, we:
- Sum edge importance scores per hub node (category or city)
- Normalise to a 0–1 node importance score
- The Streamlit UI can then colour hub nodes by this importance

Node-level feature masks are also saved for the customer node itself.

In [ ]:
gnn_jsonl_path = OUTPUTS_DIR / 'gnn_explainer_webapp.jsonl'
gnn_records = []

# Select customers: top 3 from each risk band for explanation generation
risk_col = 'fraud_score' if 'fraud_score' in model_output.columns else 'scarcity_anchor_ensemble_prob'
scores = pd.to_numeric(model_output[risk_col], errors='coerce').fillna(0.0).clip(0.0, 1.0)
df_scores = model_output[['customer_id']].copy()
df_scores['risk_score'] = scores

low_df = df_scores[df_scores['risk_score'] <= 0.33].sort_values(['risk_score', 'customer_id'])
mid_df = df_scores[(df_scores['risk_score'] > 0.33) & (df_scores['risk_score'] <= 0.66)].sort_values(['risk_score', 'customer_id'])
high_df = df_scores[df_scores['risk_score'] > 0.66].sort_values(['risk_score', 'customer_id'], ascending=[False, True])

selected = pd.concat([
    low_df.head(3).assign(risk_band='low'),
    mid_df.head(3).assign(risk_band='medium'),
    high_df.head(3).assign(risk_band='high'),
], ignore_index=True)

selected_ids = selected['customer_id'].tolist()
band_map = dict(zip(selected['customer_id'], selected['risk_band']))

# Save selected customers for reference
sel_path = OUTPUTS_DIR / 'explainability_selected_customers.json'
sel_path.write_text(json.dumps(selected.to_dict(orient='records'), indent=2))
print(f'Saved: {sel_path}')
print(f'Selected {len(selected_ids)} customers for GNN explanation:')
for _, r in selected.iterrows():
    print(f'  {r["customer_id"]} ({r["risk_band"]}) score={r["risk_score"]:.3f}')

# ── Run GNNExplainer ───────────────────────────────────────────────────────
if TORCH_GEO_OK:
    arts = load_artifacts()
    model_sage = load_sage_model(artifacts=arts)

    data = HeteroData()
    data['customer'].x = arts['x_cust']
    data['category'].x = arts['x_cat']
    data['city'].x = arts['x_city']
    data[('customer', 'purchases_at', 'category')].edge_index = arts['edge_cust_cat']
    data[('customer', 'transacts_in', 'city')].edge_index = arts['edge_cust_city']
    data = T.ToUndirected()(data)

    cust_map = arts['cust_map']
    rev_cust = {v: k for k, v in cust_map.items()}
    rev_cat = {v: k for k, v in cat_map.items()}
    rev_city = {v: k for k, v in city_map.items()}

    # Build reverse maps for category and city indices
    cat_map = arts.get('cat_map', {})
    city_map = arts.get('city_map', {})
    if not cat_map and 'cat_map' in dir(arts):
        cat_map = arts.cat_map
    if not city_map and 'city_map' in dir(arts):
        city_map = arts.city_map

    class Wrap(torch.nn.Module):
        def __init__(self, base):
            super().__init__()
            self.base = base
        def forward(self, x_dict, edge_index_dict):
            return self.base(x_dict, edge_index_dict).squeeze(-1)

    explainer = Explainer(
        model=Wrap(model_sage).eval(),
        algorithm=GNNExplainer(epochs=15),
        explanation_type='model',
        node_mask_type='attributes',
        edge_mask_type='object',
        model_config=dict(mode='binary_classification', task_level='node', return_type='raw'),
    )

    for cid in selected_ids:
        rec = {
            'customer_id': cid,
            'risk_band': band_map[cid],
            'hub_importance': {},  # {hub_name: importance_score}
            'customer_feature_importance': [],
            'mask_summary': {},
            'runtime_sec': None,
        }
        t0 = time.time()
        try:
            idx = int(cust_map[cid])
            seed = torch.tensor([idx], dtype=torch.long)
            sub = next(iter(NeighborLoader(data, num_neighbors=[20, 15, 10],
                                          input_nodes=('customer', seed),
                                          batch_size=1, shuffle=False, num_workers=0)))
            loc = int(torch.where(sub['customer'].n_id == idx)[0][0].item())
            exp = explainer(sub.x_dict, sub.edge_index_dict, index=loc)

            # ── Aggregate edge importance TO hub nodes ──────────────────
            hub_scores: dict[str, float] = {}

            # Category edges
            if ('customer', 'purchases_at', 'category') in sub.edge_index_dict:
                eidx = sub.edge_index_dict[('customer', 'purchases_at', 'category')]
                emask = getattr(exp, 'edge_mask_dict', {}).get(('customer', 'purchases_at', 'category'))
                if emask is not None and emask.numel() > 0:
                    # Map local dst indices → global category names
                    cat_nids = sub['category'].n_id.tolist() if hasattr(sub['category'], 'n_id') else []
                    for e in range(emask.size(0)):
                        dst_local = int(eidx[1, e].item())
                        if dst_local < len(cat_nids):
                            global_cat_idx = cat_nids[dst_local]
                            cat_name = rev_cat.get(int(global_cat_idx), f'cat_{global_cat_idx}')
                            hub_scores[cat_name] = hub_scores.get(cat_name, 0.0) + float(emask[e])

            # City edges
            if ('customer', 'transacts_in', 'city') in sub.edge_index_dict:
                eidx = sub.edge_index_dict[('customer', 'transacts_in', 'city')]
                emask = getattr(exp, 'edge_mask_dict', {}).get(('customer', 'transacts_in', 'city'))
                if emask is not None and emask.numel() > 0:
                    city_nids = sub['city'].n_id.tolist() if hasattr(sub['city'], 'n_id') else []
                    for e in range(emask.size(0)):
                        dst_local = int(eidx[1, e].item())
                        if dst_local < len(city_nids):
                            global_city_idx = city_nids[dst_local]
                            city_name = rev_city.get(int(global_city_idx), f'city_{global_city_idx}')
                            hub_scores[city_name] = hub_scores.get(city_name, 0.0) + float(emask[e])

            # Normalise hub scores to [0, 1]
            if hub_scores:
                max_score = max(hub_scores.values())
                if max_score > 0:
                    hub_scores = {k: round(v / max_score, 4) for k, v in hub_scores.items()}
            rec['hub_importance'] = dict(sorted(hub_scores.items(), key=lambda x: -x[1]))

            # ── Customer node feature importance ───────────────────────
            cm = getattr(exp, 'node_mask_dict', {}).get('customer')
            if cm is not None and cm.ndim == 2 and loc < cm.shape[0]:
                m = cm[loc]
                topf = torch.topk(m, k=min(5, m.numel()))
                rec['customer_feature_importance'] = [
                    {'feature_index': int(i), 'score': float(s)}
                    for s, i in zip(topf.values.tolist(), topf.indices.tolist())
                ]

            rec['mask_summary'] = {
                'hub_count': len(hub_scores),
                'hub_max_score': max(hub_scores.values()) if hub_scores else 0.0,
                'hub_mean_score': float(np.mean(list(hub_scores.values()))) if hub_scores else 0.0,
            }
        except Exception as e:
            rec['mask_summary'] = {'error': str(e)}

        rec['runtime_sec'] = round(time.time() - t0, 3)
        gnn_records.append(rec)
else:
    for cid in selected_ids:
        gnn_records.append({
            'customer_id': cid,
            'risk_band': band_map[cid],
            'hub_importance': {},
            'customer_feature_importance': [],
            'mask_summary': {'error': 'torch_geometric unavailable'},
            'runtime_sec': 0.0,
        })

with open(gnn_jsonl_path, 'w', encoding='utf-8') as f:
    for rec in gnn_records:
        f.write(json.dumps(rec) + '\n')

print(f'\nSaved: {gnn_jsonl_path}')
print(f'Records: {len(gnn_records)}')
for rec in gnn_records:
    hubs = list(rec['hub_importance'].keys())[:5]
    print(f'  {rec["customer_id"]} ({rec["risk_band"]}): {len(rec["hub_importance"])} hubs → {hubs}')

## 5. Autoencoder Anomalous Transactions

Extracts the most anomalous transactions for each selected customer, ranked by AE risk score.
The risk score is used by the Streamlit UI to colour-code transaction rows.

In [ ]:
ae_jsonl_path = OUTPUTS_DIR / 'ae_anomalies_webapp.jsonl'

tx = master_pool[master_pool['customer_id'].isin(selected_ids)].copy()
tx['transaction_datetime'] = pd.to_datetime(tx['transaction_datetime'], errors='coerce')

# Merge per-transaction AE scores
if 'transaction_id' in tx.columns and 'transaction_id' in ae_scores.columns:
    ae_scores['transaction_id'] = ae_scores['transaction_id'].astype(str)
    tx['transaction_id'] = tx['transaction_id'].astype(str)
    tx = tx.merge(
        ae_scores[['transaction_id', 'customer_id', 'reconstruction_error', 'ae_risk_score']],
        on=['transaction_id', 'customer_id'], how='left'
    )

if 'reconstruction_error' not in tx.columns:
    tx['reconstruction_error'] = np.nan
if 'ae_risk_score' not in tx.columns:
    tx['ae_risk_score'] = np.nan

ae_records = []
for cid in selected_ids:
    c = tx[tx['customer_id'] == cid].copy()
    c['reconstruction_error'] = pd.to_numeric(c['reconstruction_error'], errors='coerce')
    c['ae_risk_score'] = pd.to_numeric(c['ae_risk_score'], errors='coerce')
    c = c.sort_values(['ae_risk_score', 'reconstruction_error'], ascending=False).head(15)

    for _, r in c.iterrows():
        ae_records.append({
            'customer_id': cid,
            'risk_band': band_map[cid],
            'transaction_id': str(r.get('transaction_id', '')),
            'transaction_datetime': str(r.get('transaction_datetime', '')),
            'amount_cad': float(pd.to_numeric(r.get('amount_cad', 0.0), errors='coerce') or 0.0),
            'merchant_category': str(r.get('merchant_category', '')),
            'city': str(r.get('city', '')),
            'source_dataset': str(r.get('source_dataset', '')),
            'cash_indicator': int(pd.to_numeric(r.get('cash_indicator', 0), errors='coerce') or 0),
            'ecommerce_ind': int(pd.to_numeric(r.get('ecommerce_ind', 0), errors='coerce') or 0),
            'reconstruction_error': float(pd.to_numeric(r.get('reconstruction_error', 0.0), errors='coerce') or 0.0),
            'ae_risk_score': float(pd.to_numeric(r.get('ae_risk_score', 0.0), errors='coerce') or 0.0),
        })

with open(ae_jsonl_path, 'w', encoding='utf-8') as f:
    for rec in ae_records:
        f.write(json.dumps(rec) + '\n')

print('Saved:', ae_jsonl_path)
print('Rows:', len(ae_records))

## 6. Batch LLM Explanations

Generates explanation text for each customer:

- **Medium/high risk:** LLM-generated explanation (if endpoint is enabled), with deterministic fallback on failure
- **Low risk:** Deterministic fallback only (no need to waste LLM calls on low-risk customers)

The deterministic explanation uses SHAP top drivers to construct a readable sentence.

In [ ]:
explain_jsonl_path = OUTPUTS_DIR / 'customer_explanations_webapp.jsonl'

# Build lookup dicts
shap_by_cid = {r['customer_id']: r for r in all_shap_records}
gnn_by_cid  = {r['customer_id']: r for r in gnn_records}
ae_by_cid   = {}
for r in ae_records:
    ae_by_cid.setdefault(r['customer_id'], []).append(r)

def deterministic_explanation(cid: str, band: str, pred: float, top_shap_items: list[dict],
                              top_ae: dict | None, gnn_hubs: dict[str, float]) -> str:
    """Build a readable explanation from structured data (no LLM)."""
    parts = [f"Customer {cid[:16]} is {band}-risk with a model score of {pred:.1%}."]

    # Top SHAP drivers
    if top_shap_items:
        drivers = []
        for d in top_shap_items[:3]:
            desc = d.get('description', d.get('feature', 'unknown'))
            sv = d.get('shap_value', 0)
            if sv > 0:
                drivers.append(f"{desc} (risk +{abs(sv):.4f})")
            else:
                drivers.append(f"{desc} (risk -{abs(sv):.4f})")
        if drivers:
            parts.append("Key risk drivers: " + "; ".join(drivers) + ".")

    # GNN hub importance
    if gnn_hubs:
        top_hubs = sorted(gnn_hubs.items(), key=lambda x: -x[1])[:3]
        hub_text = ", ".join(f"{h} ({s:.0%})" for h, s in top_hubs)
        parts.append(f"Most influential transaction connections: {hub_text}.")

    # AE anomaly
    if top_ae:
        amt = float(top_ae.get('amount_cad', 0) or 0)
        cat = top_ae.get('merchant_category', '?')
        city = top_ae.get('city', '?')
        ae_score = float(top_ae.get('ae_risk_score', 0) or 0)
        parts.append(f"Most anomalous transaction: ${amt:.2f} at {cat} in {city} (anomaly score: {ae_score:.3f}).")
    else:
        parts.append("No highly anomalous transactions detected.")

    parts.append(
        "Recommended action: Review the customer's transaction history and KYC profile "
        "to determine if further investigation is warranted."
    )
    return ' '.join(parts)

def build_llm_prompt(cid: str, band: str, pred: float, top_shap_items: list[dict],
                     top_edges_text: str, top_ae_text: str) -> str:
    """Assemble the structured evidence for the LLM."""
    top_shap_text = '; '.join(
        f"{x.get('description', x['feature'])} ({x['shap_value']:+.4f})"
        for x in top_shap_items[:3]
    ) or 'no SHAP drivers'

    evidence = (
        f"Customer: {cid}\n"
        f"Risk band: {band}\n"
        f"Model fraud probability: {pred:.4f}\n"
        f"Top SHAP drivers: {top_shap_text}\n"
        f"Graph (GNN) connections: {top_edges_text}\n"
        f"Autoencoder anomalies: {top_ae_text}"
    )
    return assemble_prompt(llm_config, evidence)

def call_llm_with_retry(prompt: str, max_attempts: int = 3) -> tuple[str | None, str]:
    """Call LLM with retry logic. Returns (text, status)."""
    for attempt in range(max_attempts):
        try:
            result = call_llm(prompt, config=llm_config)
            if result:
                return result, 'ok'
        except Exception as e:
            if attempt < max_attempts - 1:
                time.sleep(0.5 * (attempt + 1))
            else:
                return None, f'error: {e}'
    return None, 'error: empty response'

# ── Generate explanations for each selected customer ───────────────────────
explanation_records = []

for cid in selected_ids:
    band = band_map[cid]
    s = shap_by_cid.get(cid, {})
    g = gnn_by_cid.get(cid, {})
    a = ae_by_cid.get(cid, [])

    pred = float(s.get('lgb_fraud_prob', 0.0))
    # Filter to interpretable features only (no latent PCA axes)
    top_shap_all = s.get('top_shap_features', [])
    top_shap_items = [d for d in top_shap_all
                      if not d['feature'].startswith('emb_pca_')][:5]

    # GNN hub importance text
    hubs = g.get('hub_importance', {})
    if hubs:
        top_hubs = sorted(hubs.items(), key=lambda x: -x[1])[:3]
        top_edges_text = '; '.join(f"{h} importance={s:.2f}" for h, s in top_hubs)
    else:
        top_edges_text = 'no significant graph connections'

    # AE anomaly text
    top_ae = a[0] if a else None
    if top_ae:
        top_ae_text = (
            f"{top_ae.get('merchant_category','?')} in {top_ae.get('city','?')} "
            f"amount ${float(top_ae.get('amount_cad', 0) or 0):.2f} "
            f"ae_risk={float(top_ae.get('ae_risk_score', 0) or 0):.4f}"
        )
    else:
        top_ae_text = 'no high-anomaly transaction found'

    # Decide: LLM or deterministic?
    use_llm = llm_config.enabled and (band in ('medium', 'high'))

    if use_llm:
        prompt = build_llm_prompt(cid, band, pred, top_shap_items, top_edges_text, top_ae_text)
        explanation_text, llm_status = call_llm_with_retry(prompt)
        if not explanation_text:
            llm_status = 'fallback'
            explanation_text = deterministic_explanation(cid, band, pred, top_shap_items, top_ae, hubs)
    else:
        llm_status = 'deterministic' if not llm_config.enabled else 'low_risk_deterministic'
        explanation_text = deterministic_explanation(cid, band, pred, top_shap_items, top_ae, hubs)

    explanation_records.append({
        'customer_id': cid,
        'risk_band': band,
        'model_fraud_prob': pred,
        'explanation_text': explanation_text,
        'llm_status': llm_status,
    })

with open(explain_jsonl_path, 'w', encoding='utf-8') as f:
    for rec in explanation_records:
        f.write(json.dumps(rec) + '\n')

print(f'Saved: {explain_jsonl_path}')
print(f'Records: {len(explanation_records)}')
for rec in explanation_records:
    print(f'\n{"="*60}')
    print(f'Customer: {rec["customer_id"]}  ({rec["risk_band"]})  Status: {rec["llm_status"]}')
    print(rec['explanation_text'])

## 7. Save Unified Web App Bundle

Writes a CSV + JSONL bundle that the Streamlit app reads. This contains the combined evidence for all customers in one place.

In [ ]:
bundle_jsonl_path = OUTPUTS_DIR / 'explainability_bundle_webapp.jsonl'
bundle_csv_path   = OUTPUTS_DIR / 'explainability_bundle_webapp.csv'

# Re-read all JSONL outputs for consistency
def _read_jsonl(path):
    with open(path) as fh:
        return [json.loads(line) for line in fh if line.strip()]

shap_data = {r['customer_id']: r for r in all_shap_records}
ae_data   = {}
for r in ae_records:
    ae_data.setdefault(r['customer_id'], []).append(r)
exp_data  = {r['customer_id']: r for r in _read_jsonl(explain_jsonl_path)}
gnn_data  = {r['customer_id']: r for r in _read_jsonl(gnn_jsonl_path)}

sel_scores = selected.set_index('customer_id')['risk_score'].to_dict()

rows = []
for cid in selected_ids:
    s = shap_data.get(cid, {})
    risk_score = s.get('lgb_fraud_prob', sel_scores.get(cid, 0.0))
    top_shap_feats = s.get('top_shap_features', [])
    top_shap_name = top_shap_feats[0]['feature'] if top_shap_feats else ''

    ae_list = ae_data.get(cid, [])
    top_txn = ae_list[0] if ae_list else {}

    c_exp = exp_data.get(cid, {})

    rows.append({
        'customer_id': cid,
        'risk_band': band_map[cid],
        'risk_score': float(risk_score),
        'top_shap_feature': top_shap_name,
        'top_anomalous_transaction_id': str(top_txn.get('transaction_id', '')),
        'top_anomalous_amount_cad': float(top_txn.get('amount_cad', 0.0) or 0.0),
        'explanation_text': c_exp.get('explanation_text', ''),
        'llm_status': c_exp.get('llm_status', ''),
    })

bundle_df = pd.DataFrame(rows)
bundle_df.to_csv(bundle_csv_path, index=False)

with open(bundle_jsonl_path, 'w', encoding='utf-8') as f:
    for r in rows:
        f.write(json.dumps(r) + '\n')

print('Saved:', bundle_jsonl_path)
print('Saved:', bundle_csv_path)
print(f'\nBundle preview ({len(rows)} customers):')
display(bundle_df[['customer_id', 'risk_band', 'risk_score', 'top_shap_feature', 'explanation_text']])